# 01 — Embedding Baseline: MobileNetV3 + ArcFace (FP32)

Visualización del espacio latente FP32 vía t-SNE, métricas topológicas
baseline (Bloque E: kNN accuracy, linear probe) y análisis de la geometría
inicial antes de cuantización.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from geoquant.models.backbone import build_backbone
from geoquant.data.dataset import get_dataloaders
from geoquant.evaluation.suite import EvaluationSuite
from geoquant.evaluation.embeddings import extract_and_save, load_embeddings
from geoquant.utils.reproducibility import seed_everything

seed_everything(42)

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

In [ ]:
device = torch.device('cpu')
_, val_loader = get_dataloaders(config)

EMB_PATH = '../outputs/embeddings/emb_fp32.pt'

# Si ya existen los embeddings en disco, los cargamos directamente
from pathlib import Path
if Path(EMB_PATH).exists():
    print(f'Cargando embeddings desde disco: {EMB_PATH}')
    emb, labels = load_embeddings(EMB_PATH)
else:
    print('Extrayendo embeddings (primera vez, se guardarán en disco)...')
    emb, labels = extract_and_save(
        ckpt_path='../outputs/checkpoints/best_fp32.pth',
        output_path=EMB_PATH,
        config=config,
        dataloader=val_loader,
        device=device,
    )

print(f'Embeddings: {emb.shape}')

In [ ]:
# t-SNE sobre las primeras 20 clases para visualización
mask = labels < 20
emb_sub = emb[mask].numpy()
labels_sub = labels[mask].numpy()

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
proj = tsne.fit_transform(emb_sub)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(proj[:, 0], proj[:, 1], c=labels_sub, cmap='tab20', alpha=0.6, s=8)
plt.colorbar(scatter)
plt.title('t-SNE — Espacio latente FP32 (primeras 20 clases CUB-200)')
plt.tight_layout()
plt.show()

In [ ]:
# Métricas baseline
suite = EvaluationSuite(k_neighbors=10)
metrics = suite.run_single(emb, labels)
print('Métricas baseline FP32:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')